# Week 57, Day 2: Kernel ridge regression

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w057/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Kernel ridge from scratch

Write rbf(A, B, ell) for rows of A and B, krr_fit(X, y, ell, lam) returning alpha, and krr_predict(X_train, alpha, X_new, ell). Fit the tuning curve and report the test MSE against the noise variance of 4.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(572)
tuning = lambda th: 10 + 12 * np.exp(-((th - 0.6) ** 2) / 0.08) + 6 * np.exp(-((th + 0.9) ** 2) / 0.2)
Xo = r.uniform(-np.pi / 2, np.pi / 2, (80, 1)); yo = tuning(Xo[:, 0]) + 2.0 * r.normal(size=80)
Xt = np.linspace(-np.pi / 2, np.pi / 2, 400)[:, None]; yt = tuning(Xt[:, 0]) + 2.0 * r.normal(size=400)

def rbf(A, B, ell):
    d2 = (A ** 2).sum(1)[:, None] - 2 * A @ B.T + (B ** 2).sum(1)[None, :]
    return ...

def krr_fit(X, y, ell, lam):
    return ...

def krr_predict(X, alpha, Xn, ell):
    return rbf(Xn, X, ell) @ alpha

ym = yo.mean()
alpha = krr_fit(Xo, yo - ym, 0.3, 0.5)
pred = krr_predict(Xo, alpha, Xt, 0.3) + ym
mse_test = np.mean((pred - yt) ** 2)
print(f"test MSE {mse_test:.2f} (noise variance 4.00)")
plt.plot(Xo[:, 0], yo, "k.", label="trials"); plt.plot(Xt[:, 0], tuning(Xt[:, 0]), "k--", label="true"); plt.plot(Xt[:, 0], pred, label="kernel ridge")
plt.xlabel("orientation (rad)"); plt.ylabel("rate (Hz)"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_kernel():
    "rbf(A, B, ell) returns exp(-|a - b|² / (2 ell²)) for every pair"
    A = np.array([[0.0], [1.0]]); assert np.allclose(rbf(A, A, 1.0), [[1, np.exp(-0.5)], [np.exp(-0.5), 1]])
def test_fit():
    "krr_fit solves (K + lam I) alpha = y and krr_predict uses the cross-kernel"
    al = krr_fit(Xo, yo, 0.3, 0.1)
    assert np.allclose((rbf(Xo, Xo, 0.3) + 0.1 * np.eye(len(yo))) @ al, yo)
def test_good():
    "the fit with ell = 0.3 and lam = 0.5 has test MSE within 50% of the noise variance"
    assert mse_test < 1.5 * 4.0

test_kernel()
test_fit()
test_good()
print("All checks passed")

### Exercise: Cross-validating two hyperparameters

For every length-scale and penalty on the grids, compute the five-fold cross-validated MSE on the 80 training trials. Refit with the best pair and evaluate on the test set, and compare with the best test MSE on the grid.

In [ ]:
ells, lams = np.array([0.03, 0.08, 0.15, 0.3, 0.6, 1.2]), np.logspace(-2, 1.5, 8)
folds = np.array_split(r.permutation(80), 5)
cv = np.zeros((len(ells), len(lams)))
for i, ell in enumerate(ells):
    for j, lam in enumerate(lams):
        e = 0.0
        for te in folds:
            tr = np.setdiff1d(np.arange(80), te)
            m = yo[tr].mean()
            a = krr_fit(Xo[tr], yo[tr] - m, ell, lam)
            e += ...
        cv[i, j] = e / 80
i, j = np.unravel_index(np.argmin(cv), cv.shape)
best_ell, best_lam = ells[i], lams[j]
tmse = lambda ell, lam: np.mean((krr_predict(Xo, krr_fit(Xo, yo - ym, ell, lam), Xt, ell) + ym - yt) ** 2)
mse_best = tmse(best_ell, best_lam)
oracle = min(tmse(e_, l_) for e_ in ells for l_ in lams)
print(f"chosen length-scale {best_ell}, penalty {best_lam:.2f}; test MSE {mse_best:.2f} (best on grid {oracle:.2f})")
for i_, ell in enumerate(ells):
    print(f"ell = {ell:4.2f}: best cross-validated MSE {cv[i_].min():.2f}")

In [ ]:
import numpy as np
def test_grid():
    "cv has one entry per length-scale and penalty"
    assert cv.shape == (len(ells), len(lams))
def test_choice():
    "the chosen length-scale is neither the smallest nor the largest on the grid"
    assert ells[0] < best_ell < ells[-1]
def test_test():
    "the cross-validated choice has test MSE within 15% of the best on the grid"
    assert mse_best <= 1.15 * oracle

test_grid()
test_choice()
test_test()
print("All checks passed")

### Exercise: How many random features?

Write rff_map(X, W, b). For 2,000 training trials of the tuning curve, fit exact kernel ridge and ridge on D random features for several D, and record the RMS difference between their predictions as a fraction of the standard deviation of the exact predictions.

In [ ]:
def rff_map(X, W, b):
    return ...

X2 = r.uniform(-np.pi / 2, np.pi / 2, (2000, 1)); y2 = tuning(X2[:, 0]) + 2.0 * r.normal(size=2000); m2 = y2.mean()
exact = krr_predict(X2, krr_fit(X2, y2 - m2, 0.15, 1.0), Xt, 0.15)
Ds = [10, 30, 100, 300, 1000, 3000]
gap = []
for D in Ds:
    W = r.normal(size=(D, 1)) / 0.15; b = r.uniform(0, 2 * np.pi, D)
    Z, Zt = rff_map(X2, W, b), rff_map(Xt, W, b)
    w = np.linalg.solve(Z.T @ Z + 1.0 * np.eye(D), Z.T @ (y2 - m2))
    gap.append(np.sqrt(np.mean((Zt @ w - exact) ** 2)) / exact.std())
    print(f"D = {D:5d}: RMS difference from exact kernel ridge {gap[-1]:.4f} of the prediction spread")

In [ ]:
import numpy as np
def test_z():
    "rff_map returns n x D features whose inner products approximate the kernel"
    g = np.random.default_rng(0); Wf = g.normal(size=(5000, 1)) / 0.3; bf = g.uniform(0, 2 * np.pi, 5000)
    Z = rff_map(Xo[:5], Wf, bf)
    assert Z.shape == (5, 5000) and np.abs(Z @ Z.T - rbf(Xo[:5], Xo[:5], 0.3)).max() < 0.05
def test_converge():
    "the gap to exact kernel ridge falls as D grows, and is below 1% of the spread of the predictions by D = 3000"
    assert gap[-1] < gap[0] and gap[Ds.index(3000)] < 0.01

test_z()
test_converge()
print("All checks passed")